# 📌 MODUL 05 — Pertemuan 06: Format Parquet, JSON, & Pipeline ETL Terdistribusi
> **Tujuan Pembelajaran:** Menguasai keunggulan format **Apache Parquet** dibanding CSV, memahami cara kerja **Data Partitioning** (*Partition Pruning*), dan membangun pipeline **ETL** (*Extract, Transform, Load*) yang menangani missing value secara aman.

## 🥊 1. Mengapa Apache Parquet Mengalahkan CSV di Big Data?

1. **Columnar Storage (Berorientasi Kolom):**  
   Data disimpan per kolom, bukan per baris. Kueri yang hanya butuh kolom `kategori` dan `harga` hanya akan membaca 2 kolom tersebut dari disk (**Column Pruning**). Kolom lain diabaikan!
2. **Kompresi Tinggi (Snappy):**  
   Parquet secara native terkompresi, menghemat ruang disk 70% hingga 80% dibanding CSV.
3. **Penyimpanan Skema:**  
   Tipe data (integer, string, timestamp) tersimpan di metadata berkas footer. Tidak perlu inferSchema yang lambat!
4. **Predicate Pushdown:**  
   Penyaringan filter (`WHERE / filter`) dievaluasi di level storage sebelum data dimuat ke memori RAM.

## ⚙️ 2. Setup Environment & Inisialisasi SparkSession

In [ ]:
import os
import sys
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Konfigurasi isolasi lingkungan native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Responsi-Pertemuan06") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ SparkSession Aktif!")

## 📦 3. Menyiapkan 3 Sumber Data Mentah (Skenario ETL)
Di industri nyata, data datang dari berbagai format:
- Sumber 1: Transaksi Harian (CSV)
- Sumber 2: Katalog Master Produk (JSON Lines)
- Sumber 3: Rating Ulasan Pembeli (CSV)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(33)

# 1. Sumber Transaksi (CSV)
df_trx_raw = pd.DataFrame({
    "order_id": [f"ORD-{i:03d}" for i in range(1, 16)],
    "product_id": np.random.choice(["PRD-A", "PRD-B", "PRD-C", "PRD-D", "PRD-E"], size=15),
    "unit_terjual": np.random.randint(1, 6, size=15)
})
df_trx_raw.to_csv("raw_transaksi.csv", index=False)

# 2. Sumber Produk (JSON Lines)
produk_list = [
    {"product_id": "PRD-A", "nama_produk": "Wireless Earbuds", "kategori": "Elektronik", "harga": 350000},
    {"product_id": "PRD-B", "nama_produk": "Kemeja Flanel", "kategori": "Fashion", "harga": 180000},
    {"product_id": "PRD-C", "nama_produk": "Kopi Arabika 250g", "kategori": "Makanan", "harga": 75000},
    {"product_id": "PRD-D", "nama_produk": "Pembersih Lantai", "kategori": "Rumah Tangga", "harga": 25000},
    {"product_id": "PRD-E", "nama_produk": "Serum Vitamin C", "kategori": "Kesehatan", "harga": 120000},
]
with open("raw_produk.json", "w") as f:
    for item in produk_list:
        f.write(json.dumps(item) + "\n")

# 3. Sumber Ulasan (CSV) — Sengaja ada yang tidak mengulas (hanya 10 baris dari 15 transaksi)
df_review_raw = pd.DataFrame({
    "order_id": [f"ORD-{i:03d}" for i in [1, 2, 4, 5, 7, 8, 10, 11, 13, 14]],
    "rating": [5, 4, 5, 3, 4, 5, 2, 4, 5, 4]
})
df_review_raw.to_csv("raw_ulasan.csv", index=False)

print("✅ Tiga sumber data mentah berhasil dibuat di disk lokal.")

## 🚀 4. Membangun Pipeline ETL Resmi

### Tahap 1: EXTRACT (Membaca Ketiga Sumber Data)

In [ ]:
df_transaksi = spark.read.csv("raw_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("raw_produk.json")
df_ulasan = spark.read.csv("raw_ulasan.csv", header=True, inferSchema=True)

print(f"Transaksi : {df_transaksi.count()} baris")
print(f"Produk    : {df_produk.count()} baris")
print(f"Ulasan    : {df_ulasan.count()} baris")

### Tahap 2: TRANSFORM (Penggabungan, Handling Null, & Pengayaan Data)
1. **Left Join ke Ulasan:** Menjamin 5 transaksi yang belum diulas **tidak terhapus** (*zero data loss*).
2. **Inner Join ke Produk:** Melengkapi nama produk, kategori, dan harga.
3. **Imputasi Nilai Null:** Rating kosong diisi angka `0` (*sentinel value* untuk membedakan ulasan bintang 1).

In [ ]:
# A. Multi-join
df_gabung = df_transaksi \
    .join(df_ulasan, on="order_id", how="left") \
    .join(df_produk, on="product_id", how="inner")

# B. Hitung total pendapatan
df_gabung = df_gabung.withColumn("total_pendapatan", F.col("unit_terjual") * F.col("harga"))

# C. Tambahkan flag boolean & imputasi null
df_enriched = df_gabung.withColumn("ada_ulasan", F.col("rating").isNotNull())
df_final = df_enriched.na.fill({"rating": 0})

print("Cuplikan Hasil Transformasi (Data Bersih):")
df_final.show(6)

### Tahap 3: LOAD (Menyimpan ke Format Parquet Terpartisi)
Menggunakan `partitionBy("kategori")` agar folder fisik diorganisir per kategori:

In [ ]:
target_parquet = "demo_hasil_etl_parquet"

# Simpan ke Parquet terpartisi kategori
df_final.write.mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(target_parquet)

print(f"✅ Data berhasil disimpan ke: {target_parquet}")

# Cek struktur subfolder partisi di sistem berkas
!ls -la demo_hasil_etl_parquet

## 📈 5. Evaluasi Insight Bisnis: Rasio Transaksi Berulasan per Kategori
Mencari kategori mana yang pembelinya paling malas memberikan ulasan:

In [ ]:
df_insight = df_final.groupBy("kategori").agg(
    F.count("order_id").alias("total_transaksi"),
    F.sum(F.when(F.col("ada_ulasan") == True, 1).otherwise(0)).alias("transaksi_diulas")
).withColumn(
    "persentase_ulasan",
    F.round((F.col("transaksi_diulas") / F.col("total_transaksi")) * 100, 2)
).orderBy(F.col("persentase_ulasan").asc())

print("Rasio Ulasan per Kategori Produk:")
df_insight.show()

## 🧹 6. Bersih-bersih & Menutup Sesi

In [ ]:
# Hapus berkas sampah pengujian
!rm -rf raw_transaksi.csv raw_produk.json raw_ulasan.csv demo_hasil_etl_parquet
spark.stop()
print("🛑 SparkSession ditutup dan berkas sampah dibersihkan.")